# CV-localization: interactive explorationThis notebook is for looking at one cycle by hand: the landscape, the twocurves, the search trace, and the discrete Cholesky version. It reads nothingfrom `results/`, so it works before any experiment has been run.

In [ ]:
import sys, numpy as np, matplotlib.pyplot as pltsys.path[:0] = ["/work"]import cvlocfrom cvloc import CycleConfig, make_cycle, EnsembleSpace, truth_optimum, cv_sweep, select_radiusprint("cvloc", cvloc.__version__)

## One cycleThe ensemble is climatological: members are widely separated snapshots of a longfree run, so the background error is large and the sample covariance carries thefull sampling noise of a raw ensemble. That is the regime in which localizationis decisive.

In [ ]:
cfg = CycleConfig(n=40, N=20)cycle = make_cycle(cfg, seed=42)space = EnsembleSpace(cycle)print(f"background RMSE      {cycle.background_rmse():.3f}")print(f"no localization      {space.rmse(space.analysis_mean(None)):.3f}")print(f"r = 2                {space.truth_error(np.full(40, 2.0)):.3f}")

## The two curvesThe truth-based curve is what an oracle would minimize; the cross-validated oneis what the method actually minimizes. The claim of the paper is that they areminimized in the same region.

In [ ]:
to = truth_optimum(space, 0.2, 20.0, 120)cs = cv_sweep(space, folds=10, lo=0.2, hi=20.0, n_points=120)fig, ax = plt.subplots(figsize=(6, 4))ax.plot(to["grid"], to["error"], "k-", label="analysis RMSE (truth)")ax.set_xscale("log"); ax.set_xlabel("uniform radius $r$"); ax.set_ylabel("RMSE")ax2 = ax.twinx(); ax2.plot(cs["grid"], cs["cost"], "--", color="tab:blue"); ax2.set_ylabel("CV cost", color="tab:blue")ax.axvline(to["r_opt"], color="k", ls=":"); ax.axvline(cs["r_opt"], color="tab:blue", ls=":")ax.set_title(f"oracle r = {to['r_opt']:.2f},  CV r = {cs['r_opt']:.2f}")plt.show()penalty = 100 * (space.truth_error(np.full(40, cs["r_opt"])) / to["error_opt"] - 1)print(f"spread across the range: {100*(to['error_worst']/to['error_opt']-1):.0f}%")print(f"penalty of the CV choice: {penalty:.1f}%")

## The searchChange `gamma` and watch the trace. On one free radius the landscape is smoothand almost anything converges, which is why the interesting comparisons live inthe multi-radius and Cholesky cases.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))for gamma in (0.01, 0.1, 1.0):    params = {**cvloc.defaults_for("fpa"), "gamma": gamma}    theta, r, obj, info = select_radius(space, optimizer="fpa", budget=120, seed=1, opt_params=params)    ax.plot(obj.best_trace(), label=f"gamma={gamma}  r={r[0]:.2f}")ax.set_xlabel("unique evaluations of $J$"); ax.set_ylabel("running minimum of $J$")ax.legend(); plt.show()

## More free radii`K = 1` is nested inside every larger `K`, so the attained `J` can only go downif the search is working. If it goes up, that is a search failure, not aproperty of the parameterization.

In [ ]:
for K in (1, 2, 4, 8, 40):    kind = "uniform" if K == 1 else "blocks"    theta, r, obj, info = select_radius(space, optimizer="fpa", param_kind=kind, K=K,                                        budget=80 * max(1, K // 4), seed=3)    print(f"K={K:3d}   J={info['J']:8.4f}   RMSE={space.truth_error(r):.4f}   "          f"spread of r={np.std(r):.2f}")

## The discrete landscapeHere the radius sets the predecessor set of a regression instead of the width ofa taper, so the estimator changes only at integers and the landscape is notunimodal.

In [ ]:
from cvloc.cholesky import CholeskySpacefrom cvloc.objective import CVObjectivefrom cvloc.parameterization import make as make_paramcs_space = CholeskySpace(cycle, r_max=12)obj = CVObjective(cs_space, param=make_param("uniform", 40), folds=10,                  rng=np.random.default_rng(0), budget=None, quantize=None)radii = np.arange(1, 13)cost = np.array([obj.raw(np.array([float(r)])) for r in radii])err = np.array([cs_space.truth_error(np.full(40, float(r))) for r in radii])fig, ax = plt.subplots(figsize=(6, 4))ax.plot(radii, err, "ko-", label="RMSE (truth)")ax2 = ax.twinx(); ax2.plot(radii, cost, "s--", color="tab:blue"); ax2.set_ylabel("CV cost", color="tab:blue")ax.set_xlabel("predecessor radius $r$"); ax.set_ylabel("analysis RMSE"); ax.legend()plt.show()